# AutoSolve ML Training & Export Pipeline

This notebook runs the machine learning training pipeline for the **AutoSolve** Blender camera tracking assistant. It can be run entirely in Google Colab.

It trains three core models:
1. **Track Quality Predictor**: MLP that predicts whether an active track will survive in the next 20 frames.
2. **Settings Optimizer**: MLP that predicts expected tracking reward given footage and parameters.
3. **Region Trackability Heatmap**: Empirical frequency table weighting feature placement across the screen.

*Note: All scripts support zero-dependency fallback modes that generate heuristic defaults when PyTorch or NumPy are missing from the local developer environment. Inside this notebook/Colab environment, PyTorch and NumPy are fully used for actual model training.*

Finally, it exports these weights into JSON format for native NumPy runtime inference inside Blender.

## Step 1: Upload Project Files & Environment Setup

Since we are running in Colab, we need to upload the project code. Zip your `AutoSolve` project directory and upload the `AutoSolve.zip` to Colab, or clone the repository directly if hosted on GitHub.

If you uploaded a zip, run the code cell below to unzip it and move into the directory.

In [ ]:
# Unzip project if uploaded as zip (uncomment below if needed)
# !unzip -q AutoSolve.zip -d AutoSolve
# %cd AutoSolve

# Check files in current directory
!ls -la

Verify PyTorch and NumPy installations (pre-installed in Google Colab):

In [ ]:
import torch
import numpy as np
print(f"PyTorch Version: {torch.__version__}")
print(f"NumPy Version:   {np.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

## Step 2: Prepare Datasets

Run `prepare_dataset.py`. If you have uploaded raw JSON solves, place them in `ml/data/raw/` first. If no raw data is present, the preprocessor will automatically generate a simulated dataset containing signals for testing/training.

In [ ]:
# Create directories
!mkdir -p ml/data/raw ml/data/processed ml/runs

# Run dataset preparation
!python ml/prepare_dataset.py --data-dir ml/data/raw --output-json ml/data/processed/settings_dataset.json

## Step 3: Train Track Quality Predictor

Train the 3-layer Multi-Layer Perceptron (MLP) classification network that predicts track degradation probability.

In [ ]:
!python ml/train_track_predictor.py --data-dir ml/data/raw --out-dir ml/runs/track_predictor --epochs 100

## Step 4: Train Settings Optimizer

Train the expected reward MLP model which performs preset parameter estimation.

In [ ]:
!python ml/train_settings_model.py --data-path ml/data/processed/settings_dataset.json --out-dir ml/runs/settings_optimizer --epochs 50

Evaluate the settings optimizer performance on the validation split:

In [ ]:
!python ml/evaluate_model.py --data-path ml/data/processed/settings_dataset.json --model-path ml/runs/settings_optimizer/model_meta_weights.json

## Step 5: Aggregate Region Trackability Heatmap

Compute the empirical survival weights per screen region based on footage type characteristics.

In [ ]:
!python ml/train_trackability_model.py --data-dir ml/data/raw --output-json ml/runs/region_weights.json

## Step 6: Export Models to NumPy-Compatible Formats

Export the PyTorch model weights to lightweight JSON arrays for runtime addon deployment.

In [ ]:
# Export track quality predictor weights
!python ml/export_numpy_model.py --input-json ml/runs/track_predictor/model_meta_weights.json --output-path ml/runs/track_predictor.json

# Search parameter sweep recommended defaults
!python ml/export_defaults.py --model-path ml/runs/settings_optimizer/model_meta_weights.json --output-json ml/runs/recommended_defaults.json

## Step 7: Download Exported Assets

Run this cell to download the compiled JSON weights directly to your machine. 

### Where to copy these files in your addon:
* **`track_predictor.json`** -> Copy to `autosolve/tracker/models/track_predictor.json`
* **`region_weights.json`** -> Copy to `autosolve/tracker/presets/region_weights.json`
* **`recommended_defaults.json`** -> Open and merge recommendations into `PRETRAINED_DEFAULTS` inside `autosolve/tracker/constants.py`.

In [ ]:
from google.colab import files
import os

files_to_download = [
    "ml/runs/track_predictor.json",
    "ml/runs/region_weights.json",
    "ml/runs/recommended_defaults.json"
]

for f in files_to_download:
    if os.path.exists(f):
        print(f"Downloading {f}...")
        files.download(f)
    else:
        print(f"File {f} not found!")